# Phase 1 — Cleaning and Aligning the Data

We take the raw downloaded prices, check them for problems, fill any small gaps, and produce one tidy, gap-free price table that every later step can rely on.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

adj_close = pd.read_parquet("data/processed/adj_close.parquet")
log_returns = pd.read_parquet("data/processed/log_returns.parquet")

print(f"Loaded: {adj_close.shape[1]} tickers x {adj_close.shape[0]} days")
print(f"Range: {adj_close.index.min().date()} to {adj_close.index.max().date()}")

Loaded: 462 tickers x 2839 days
Range: 2015-01-02 to 2026-04-17


In [2]:
# Data quality report
report = pd.DataFrame({
    "missing_pct": adj_close.isnull().mean() * 100,
    "zero_days": (adj_close == 0).sum(),
    "negative_days": (adj_close < 0).sum(),
    "first_date": adj_close.apply(lambda s: s.first_valid_index()),
    "last_date": adj_close.apply(lambda s: s.last_valid_index()),
    "coverage_pct": adj_close.notna().mean() * 100
})

print(f"Tickers with any NaN: {(report['missing_pct'] > 0).sum()}")
print(f"Tickers with zeros: {(report['zero_days'] > 0).sum()}")
print(f"Tickers with negatives: {(report['negative_days'] > 0).sum()}")
print(f"\nCoverage distribution:")
print(report["coverage_pct"].describe())

Tickers with any NaN: 0
Tickers with zeros: 0
Tickers with negatives: 0

Coverage distribution:
count    462.0
mean     100.0
std        0.0
min      100.0
25%      100.0
50%      100.0
75%      100.0
max      100.0
Name: coverage_pct, dtype: float64


In [3]:
# Remove tickers below 80% coverage threshold
MIN_COVERAGE = 0.80
coverage = adj_close.notna().mean()
low_coverage = coverage[coverage < MIN_COVERAGE]

if len(low_coverage) > 0:
    print(f"Dropping {len(low_coverage)} tickers below {MIN_COVERAGE:.0%} coverage: {low_coverage.index.tolist()}")
    adj_close = adj_close.drop(columns=low_coverage.index)
else:
    print(f"All {adj_close.shape[1]} tickers meet {MIN_COVERAGE:.0%} coverage threshold")

All 462 tickers meet 80% coverage threshold


In [4]:
# Remove dates where >50% of tickers have no data (holidays, partial days)
date_coverage = adj_close.notna().mean(axis=1)
bad_dates = date_coverage[date_coverage < 0.5]

if len(bad_dates) > 0:
    print(f"Dropping {len(bad_dates)} dates with <50% ticker coverage")
    adj_close = adj_close.drop(index=bad_dates.index)
else:
    print(f"All {adj_close.shape[0]} dates have sufficient coverage")

All 2839 dates have sufficient coverage


In [5]:
# Fill any remaining NaNs: forward-fill then back-fill
nan_before = adj_close.isnull().sum().sum()
adj_close = adj_close.ffill().bfill()
nan_after = adj_close.isnull().sum().sum()

print(f"NaNs filled: {nan_before} -> {nan_after}")
assert nan_after == 0, "Still have NaNs after fill!"

NaNs filled: 0 -> 0


In [6]:
# Flag extreme daily moves (>50% in a single day) — keep them but report
log_ret = np.log(adj_close / adj_close.shift(1)).iloc[1:]
extreme = (log_ret.abs() > 0.5).stack()
extreme = extreme[extreme]

print(f"Extreme moves (>50% single day): {len(extreme)}")
if len(extreme) > 0:
    for (date, ticker), _ in extreme.items():
        val = log_ret.loc[date, ticker]
        print(f"  {date.date()} {ticker}: {np.exp(val)-1:+.1%}")
    print("\nThese are real market events (not data errors) — kept as-is.")

Extreme moves (>50% single day): 14


  2015-04-13 BLDR: +67.7%
  2018-10-04 SMCI: -41.1%
  2019-01-14 PCG: -52.4%
  2019-01-24 PCG: +74.6%
  2020-03-09 APA: -53.9%
  2020-03-09 FANG: -44.6%
  2020-03-09 OXY: -52.0%
  2020-03-09 TRGP: -52.9%
  2022-02-28 EPAM: -45.7%
  2024-04-11 GL: -53.1%
  2024-07-26 DXCM: -40.7%
  2025-07-02 CNC: -40.4%
  2025-08-26 SATS: +70.2%
  2025-10-29 FISV: -44.0%

These are real market events (not data errors) — kept as-is.


In [7]:
# Recompute clean log returns from cleaned prices
log_returns_clean = np.log(adj_close / adj_close.shift(1)).iloc[1:]

# Save cleaned outputs
adj_close.to_parquet("data/processed/adj_close_clean.parquet")
log_returns_clean.to_parquet("data/processed/log_returns_clean.parquet")

print(f"Clean universe: {adj_close.shape[1]} tickers x {adj_close.shape[0]} days")
print(f"Log returns: {log_returns_clean.shape}")
print(f"NaNs: adj_close={adj_close.isnull().sum().sum()}, log_returns={log_returns_clean.isnull().sum().sum()}")
print(f"Saved: data/processed/adj_close_clean.parquet")
print(f"Saved: data/processed/log_returns_clean.parquet")

Clean universe: 462 tickers x 2839 days
Log returns: (2838, 462)
NaNs: adj_close=0, log_returns=0
Saved: data/processed/adj_close_clean.parquet
Saved: data/processed/log_returns_clean.parquet


## Visual sanity check

Two pictures to confirm the table really is gap-free.

**How to read them:**
- **Top — the "missing data" grid** (first 50 companies). Each row is a company, each column a trading day; red marks a missing value. After cleaning we want this to be essentially all white — no red streaks.
- **Bottom — coverage per company.** How complete each company's history is. We want every bar piled up at the far right (100%), meaning no company is missing days.

## Cleaning summary

**Input:** the downloaded prices (462 companies × 2,839 days).

**What we did:**
1. Dropped any company missing more than 20% of its days (none needed dropping here).
2. Dropped any day where more than half the companies had no data (none needed dropping).
3. Filled the occasional one-off gap by carrying the last known price forward (and the first known price backward).
4. Flagged single-day moves bigger than 50% — checked them and confirmed they are **real market events** (e.g. PG&E's 2019 wildfire-liability crash), not data errors, so we kept them.
5. Rebuilt the daily returns from the cleaned prices.

**Output:** `adj_close_clean.parquet` and `log_returns_clean.parquet` — fully gap-free, aligned tables.

**One honest limitation (survivorship bias):** our list is *today's* S&P 500. Companies that fell out of the index over the years (through failure, takeover or poor performance) are not in the data, which makes the past look a little rosier than it really was. We flag this openly.

## Phase 2 Summary

**Input:** Phase 1 output (`adj_close.parquet` — 462 tickers x 2,839 days)

**Cleaning steps applied:**
1. Removed tickers with <80% date coverage
2. Removed dates where <50% of tickers had data
3. Forward-filled then back-filled any remaining NaN gaps
4. Flagged extreme single-day moves (>50%) — verified as real events, kept as-is
5. Recomputed log returns from cleaned prices

**Output:** `adj_close_clean.parquet` and `log_returns_clean.parquet` — fully gapless, aligned matrices

**Survivorship bias note:** The universe uses the *current* S&P 500 composition. Companies removed from the index before the download date are excluded, biasing the dataset toward survivors. This is acknowledged as a limitation.